# Transformer-Based Semantic Game Recommender
-------------
**MSc Data Science Dissertation**  
**Author:** Vikrant Deshmukh  
**University:** University of Bristol  
**Project:** Video Game Recommendation System




This notebook develops a semantic candidate-generation model using dense transformer embeddings of cleaned Steam game descriptions.

The semantic recommender is intentionally designed to provide a signal that differs from the structured semantic recommender:

- game titles are not appended to the model input;
- genres, tags and categories are not appended;
- developer and publisher names are not appended;
- popularity and review variables are not embedded; and
- only cleaned descriptive prose is represented.

The resulting semantic candidates will later be combined with semantic and graph candidates in the hybrid recommender.

## Notebook Objectives

The notebook has six objectives:

1. construct a controlled, non-duplicated semantic description field;
2. retain broad catalogue coverage while excluding extremely sparse descriptions;
3. generate normalised dense embeddings using a pre-trained sentence transformer;
4. use GPU acceleration when CUDA is available;
5. retrieve semantically similar games using cosine similarity; and
6. export standardised Top 50 candidates for hybrid fusion.

The model is not fine-tuned on Steam data. It is used as a general-purpose semantic encoder so that its contribution can be compared fairly with lexical TF-IDF and structured semantic approaches.

## Semantic Input Design

The semantic input uses the same underlying description source as the rich-text TF-IDF baseline. This allows the two methods to be compared primarily on representation rather than on different input information.

### Included

- `about_the_game_clean` as the primary description;
- `short_description_clean` only when the primary description is empty; and
- a maximum of 250 words from the selected description.

### Excluded Fields

- `display_name` and other title fields;
- genres;
- Steam tags and tag-vote counts;
- categories and gameplay modes;
- developer and publisher names;
- release date;
- recommendations, reviews and concurrent-user measures; and
- manually repeated keywords.

The 250-word cap limits excessive marketing boilerplate and keeps input length controlled. Natural mentions that already occur inside the original description are retained because destructive phrase removal could distort legitimate meaning.

In [44]:
# Install the tested inference dependency.

import importlib.metadata
import subprocess
import sys

REQUIRED_SENTENCE_TRANSFORMERS_VERSION = "5.6.0"

try:
    installed_version = importlib.metadata.version(
        "sentence-transformers"
    )
except importlib.metadata.PackageNotFoundError:
    installed_version = None

if installed_version != REQUIRED_SENTENCE_TRANSFORMERS_VERSION:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            f"sentence-transformers=={REQUIRED_SENTENCE_TRANSFORMERS_VERSION}",
        ]
    )

print(
    "sentence-transformers version:",
    REQUIRED_SENTENCE_TRANSFORMERS_VERSION,
)

sentence-transformers version: 5.6.0


In [2]:
# Core libraries

import json
import os
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import torch

from pathlib import Path
from sentence_transformers import SentenceTransformer

os.environ["TOKENIZERS_PARALLELISM"] = "false"

## 1. Dataset Loading and Observations

The notebook reuses the cleaned Steam Games Dataset 2025 produced during the data-preparation stage.

In [4]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
# Configure the project and dataset paths.

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/MSC_DISSERTATION")
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Output/semantic_outputs"

data_path = DATA_DIR / "games_clean_v1.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

games = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", games.shape)

Dataset loaded successfully.
Dataset shape: (89618, 50)


## 2. Required Column Validation

Only the identifiers, title used for lookup, and cleaned description fields are required for embedding generation. Optional catalogue fields are retained only for output inspection and later re-ranking.

In [6]:
required_columns = {
    "appid",
    "display_name",
    "short_description_clean",
    "about_the_game_clean",
}

missing_required_columns = sorted(
    required_columns.difference(
        games.columns
    )
)

if missing_required_columns:
    raise KeyError(
        "Missing required columns: "
        + ", ".join(
            missing_required_columns
        )
    )

print(
    "Required semantic columns are available."
)

Required semantic columns are available.


## 3. Game-Title Normalisation

Titles are used only for catalogue lookup. They are not included in the semantic text passed to the transformer.

In [8]:
def normalise_game_title(title):
    """Return a stable lookup key for a game title."""

    title = str(title)

    title = (
        title
        .replace("®", "")
        .replace("™", "")
        .replace("’", "'")
        .replace("‘", "'")
        .replace("`", "'")
    )

    title = unicodedata.normalize(
        "NFKD",
        title,
    )

    title = title.casefold()

    title = re.sub(
        r"[^a-z0-9]+",
        " ",
        title,
    )

    return re.sub(
        r"\s+",
        " ",
        title,
    ).strip()


games["normalised_display_name_semantic"] = (
    games["display_name"]
    .apply(
        normalise_game_title
    )
)

games[
    [
        "display_name",
        "normalised_display_name_semantic",
    ]
].head(10)

,display_name,normalised_display_name_semantic
0,Counter-Strike 2,counter strike 2
1,PUBG: BATTLEGROUNDS,pubg battlegrounds
2,Dota 2,dota 2
3,Grand Theft Auto V Legacy,grand theft auto v legacy
4,Tom Clancy's Rainbow Six® Siege,tom clancy s rainbow six siege
5,Team Fortress 2,team fortress 2
6,Terraria,terraria
7,Rust,rust
8,Garry's Mod,garry s mod
9,Apex Legends™,apex legends


## 4. Controlled Semantic Text Construction

`about_the_game_clean` is used when available. `short_description_clean` is used only as a fallback rather than being concatenated, which avoids repeating the same promotional sentence twice.

Only whitespace normalisation and an explicit word cap are applied. Structured semantic is not injected into the semantic representation.

In [9]:
MIN_DESCRIPTION_WORDS = 30
MAX_SEMANTIC_WORDS = 250

about_text = (
    games["about_the_game_clean"]
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

short_text = (
    games["short_description_clean"]
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

games["description_text"] = np.where(
    about_text.ne(""),
    about_text,
    short_text,
)

games["description_text"] = (
    pd.Series(
        games["description_text"],
        index=games.index,
        dtype="object",
    )
    .fillna("")
    .astype(str)
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
    .str.strip()
)

games["description_word_count"] = (
    games["description_text"]
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

games["semantic_text"] = (
    games["description_text"]
    .str.split()
    .str[:MAX_SEMANTIC_WORDS]
    .str.join(" ")
)

games[
    [
        "display_name",
        "description_word_count",
        "semantic_text",
    ]
].sample(20)

,display_name,description_word_count,semantic_text
2796,World Boss,312,join lannan lazarbeam eacott and harley fresh ...
77139,Ohio John: Get Out!,89,during his first archaeological excavation ohi...
55198,Anchor Up,296,anchor up is a calming strategy puzzle game ab...
57343,Team Of Titans,181,build a team of titans with endless possibilit...
74991,Another Door: Escape Room,183,can you escape in time another door is an esca...
72222,SYNTHSPACE,214,synthspace is the ultimate virtual synthesizer...
51096,MIYAMOTO S,87,miyamoto is a single player card game with a m...
68193,The little Puzzle House,52,enjoy a relaxing puzzle game and unlock rooms ...
25395,Hidden: On the trail of the Ancients,214,immersed into dark and unexplored ancient wood...
37205,ホロポップ,421,a fan game that s easy to enjoy with the simpl...


## 5. Semantic Text Coverage

Games with fewer than 30 descriptive words are **excluded** from the embedding catalogue. This removes empty and extremely sparse descriptions without requiring additional semantic to compensate for missing prose.

In [10]:
eligible_mask = (
    games["description_word_count"]
    >= MIN_DESCRIPTION_WORDS
)

semantic_games = (
    games.loc[
        eligible_mask
    ]
    .copy()
    .reset_index(
        drop=True
    )
)

total_games = len(games)
eligible_games = len(
    semantic_games
)
excluded_games = (
    total_games - eligible_games
)
coverage_percentage = (
    100 * eligible_games / total_games
)

print(
    "Total games:",
    f"{total_games:,}",
)

print(
    "Semantic-eligible games:",
    f"{eligible_games:,}",
)

print(
    "Excluded games:",
    f"{excluded_games:,}",
)

print(
    "Eligible catalogue coverage:",
    f"{coverage_percentage:.2f}%",
)

print(
    "\nDescription word-count summary:"
)

print(
    semantic_games[
        "description_word_count"
    ]
    .describe()
)

if semantic_games.empty:
    raise ValueError(
        "No games satisfy the semantic-text eligibility rule."
    )

Total games: 89,618
Semantic-eligible games: 87,568
Excluded games: 2,050
Eligible catalogue coverage: 97.71%

Description word-count summary:
count    87568.000000
mean       219.362975
std        169.263041
min         30.000000
25%        116.000000
50%        180.000000
75%        275.000000
max      11066.000000
Name: description_word_count, dtype: float64


## 5. Transformer and CUDA Configuration

The model used is `sentence-transformers/all-mpnet-base-v2`, a general-purpose dense sentence and paragraph encoder.

CUDA is used automatically when available. GPU acceleration is required only for embedding generation; recommendation retrieval uses the saved normalised embeddings.

In [11]:
MODEL_NAME = (
    "sentence-transformers/"
    "all-mpnet-base-v2"
)

DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

BATCH_SIZE = (
    64
    if DEVICE == "cuda"
    else 16
)

print(
    "Embedding device:",
    DEVICE,
)



Embedding device: cuda


In [12]:
semantic_model = SentenceTransformer(
    MODEL_NAME,
    device=DEVICE,
)

semantic_model.max_seq_length = 384

print(
    "Model loaded:",
    MODEL_NAME,
)

print(
    "Embedding dimension:",
    semantic_model.get_embedding_dimension(),
)

print(
    "Maximum token sequence length:",
    semantic_model.max_seq_length,
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded: sentence-transformers/all-mpnet-base-v2
Embedding dimension: 768
Maximum token sequence length: 384


/tmp/ipykernel_633/1242911432.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  semantic_model.get_sentence_embedding_dimension(),


## 6. Embedding Generation and Persistence

Embeddings are generated once, L2-normalised during encoding, saved as `float32`, and reused on later notebook runs.

Because embeddings are normalised, cosine similarity is equivalent to the vector dot product. The saved index file preserves the exact row-to-game mapping.

In [33]:
safe_model_name = (
    MODEL_NAME
    .split("/")[-1]
    .replace("-", "_")
)

embedding_output_path = os.path.join(
    OUTPUT_DIR,
    (
        "semantic_embeddings_"
        f"{safe_model_name}.npy"
    ),
)

embedding_index_output_path = os.path.join(
    OUTPUT_DIR,
    (
        "semantic_embedding_index_"
        f"{safe_model_name}.csv"
    ),
)

REGENERATE_EMBEDDINGS = False

semantic_embedding_index = (
    semantic_games[
        [
            "appid",
            "display_name",
            "description_word_count",
        ]
    ]
    .copy()
)

semantic_embedding_index.insert(
    0,
    "model_row",
    np.arange(
        len(
            semantic_embedding_index
        )
    ),
)

In [35]:
def saved_embeddings_are_compatible():
    """Check whether saved embeddings match the current semantic catalogue."""

    if not (
        os.path.exists(
            embedding_output_path
        )
        and os.path.exists(
            embedding_index_output_path
        )
    ):
        return False

    saved_index = pd.read_csv(
        embedding_index_output_path
    )

    if len(saved_index) != len(
        semantic_embedding_index
    ):
        return False

    current_appids = (
        semantic_embedding_index[
            "appid"
        ]
        .astype(str)
        .to_numpy()
    )

    saved_appids = (
        saved_index[
            "appid"
        ]
        .astype(str)
        .to_numpy()
    )

    if not np.array_equal(
        current_appids,
        saved_appids,
    ):
        return False

    saved_embeddings = np.load(
        embedding_output_path,
        mmap_mode="r",
    )

    expected_shape = (
        len(
            semantic_embedding_index
        ),
        semantic_model.get_embedding_dimension(),
    )

    return (
        saved_embeddings.shape
        == expected_shape
    )

In [36]:
if (
    not REGENERATE_EMBEDDINGS
    and saved_embeddings_are_compatible()
):
    semantic_embeddings = np.load(
        embedding_output_path
    )

    print(
        "Loaded compatible saved embeddings."
    )

else:
    semantic_embeddings = (
        semantic_model.encode(
            semantic_games[
                "semantic_text"
            ]
            .tolist(),
            batch_size=BATCH_SIZE,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,
            device=DEVICE,
        )
    )

    semantic_embeddings = (
        semantic_embeddings
        .astype(
            np.float32,
            copy=False,
        )
    )

    np.save(
        embedding_output_path,
        semantic_embeddings,
    )

    semantic_embedding_index.to_csv(
        embedding_index_output_path,
        index=False,
    )

    print(
        "Generated and saved semantic embeddings."
    )

print(
    "Semantic embedding shape:",
    semantic_embeddings.shape,
)

print(
    "Embedding dtype:",
    semantic_embeddings.dtype,
)

Loaded compatible saved embeddings.
Semantic embedding shape: (87568, 768)
Embedding dtype: float32


In [37]:
expected_embedding_shape = (
    len(semantic_games),
    semantic_model.get_embedding_dimension(),
)

if (
    semantic_embeddings.shape
    != expected_embedding_shape
):
    raise ValueError(
        "Semantic embedding shape does not match the eligible catalogue."
    )

if not np.isfinite(
    semantic_embeddings
).all():
    raise ValueError(
        "Semantic embeddings contain non-finite values."
    )

embedding_norms = np.linalg.norm(
    semantic_embeddings,
    axis=1,
)

if not np.allclose(
    embedding_norms,
    1.0,
    atol=1e-3,
):
    raise ValueError(
        "Semantic embeddings are not L2-normalised."
    )

print(
    "Embedding validation passed."
)

Embedding validation passed.


## 7. Eligible Title Mapping

Only games represented in the embedding matrix can be queried. Duplicate normalised titles are resolved by retaining the record with the highest Steam recommendation count when that field exists.

In [17]:
semantic_games[
    "normalised_display_name_semantic"
] = (
    semantic_games[
        "display_name"
    ]
    .apply(
        normalise_game_title
    )
)

if "recommendations" in semantic_games.columns:
    title_mapping_source = (
        semantic_games
        .sort_values(
            "recommendations",
            ascending=False,
            na_position="last",
        )
    )
else:
    title_mapping_source = semantic_games

semantic_title_to_model_row = {}

for model_row, title_key in zip(
    title_mapping_source.index,
    title_mapping_source[
        "normalised_display_name_semantic"
    ],
):
    if not title_key:
        continue

    if (
        title_key
        not in semantic_title_to_model_row
    ):
        semantic_title_to_model_row[
            title_key
        ] = int(
            model_row
        )

assert semantic_title_to_model_row, (
    "The semantic title mapping was not created."
)

print(
    "Unique semantic titles mapped:",
    f"{len(semantic_title_to_model_row):,}",
)

Unique semantic titles mapped: 85,685


In [18]:
def search_game_titles_semantic(
    search_text,
    limit=20,
):
    """Search semantic-eligible titles using token-based matching."""

    query_key = normalise_game_title(
        search_text
    )

    query_tokens = set(
        query_key.split()
    )

    if not query_tokens:
        return []

    title_tokens = (
        semantic_games[
            "normalised_display_name_semantic"
        ]
        .astype(str)
        .str.split()
    )

    matches = title_tokens.apply(
        lambda tokens: (
            query_tokens
            .issubset(
                set(tokens)
            )
        )
    )

    return (
        semantic_games.loc[
            matches,
            "display_name",
        ]
        .drop_duplicates()
        .head(limit)
        .tolist()
    )


search_game_titles_semantic(
    "assassin creed"
)

["Assassin's Creed® Odyssey",
 "Assassin's Creed® Origins",
 "Assassin's Creed® Unity",
 "Assassin's Creed 2",
 "Assassin's Creed® Syndicate",
 "Assassin's Creed Valhalla",
 'Assassin’s Creed® Brotherhood',
 "Assassin's Creed™: Director's Cut Edition",
 'Assassin’s Creed® Rogue',
 "Assassin's Creed® Revelations",
 'Assassin’s Creed® III',
 "Assassin's Creed® III Remastered",
 "Assassin's Creed Mirage",
 'Assassin’s Creed® Chronicles: China',
 'Assassin’s Creed® Liberation HD',
 "Assassin's Creed Freedom Cry",
 'Assassin’s Creed® Chronicles: Russia',
 'Assassin’s Creed® Chronicles: India']

## 8. Semantic Recommendation Function

Each query game uses its stored normalised embedding. Similarity against the full semantic catalogue is calculated through a dot product.

The function returns semantic relevance only. Popularity and quality fields are attached for later analysis but do not alter semantic ranking in this notebook.

In [19]:
def recommend_semantic_v1(
    game_title,
    top_n=10,
    candidate_pool=50,
):
    """Recommend games using transformer-based semantic similarity."""

    if top_n < 1:
        raise ValueError(
            "top_n must be at least 1."
        )

    if candidate_pool < top_n:
        raise ValueError(
            "candidate_pool must be greater than or equal to top_n."
        )

    title_key = normalise_game_title(
        game_title
    )

    if (
        title_key
        not in semantic_title_to_model_row
    ):
        possible_matches = (
            search_game_titles_semantic(
                game_title,
                limit=10,
            )
        )

        raise ValueError(
            f"Game title not found or not semantic-eligible: {game_title}. "
            f"Possible matches: {possible_matches}"
        )

    query_model_row = (
        semantic_title_to_model_row[
            title_key
        ]
    )

    query_row = semantic_games.iloc[
        query_model_row
    ]

    query_appid = query_row[
        "appid"
    ]

    query_embedding = (
        semantic_embeddings[
            query_model_row
        ]
    )

    similarity_scores = (
        semantic_embeddings
        @ query_embedding
    )

    similarity_scores = (
        np.asarray(
            similarity_scores,
            dtype=np.float32,
        )
    )

    similarity_scores[
        query_model_row
    ] = -np.inf

    available_candidates = (
        len(semantic_games) - 1
    )

    retrieval_count = min(
        max(
            candidate_pool,
            top_n,
        ),
        available_candidates,
    )

    if retrieval_count < 1:
        return pd.DataFrame()

    candidate_indices = np.argpartition(
        -similarity_scores,
        retrieval_count - 1,
    )[
        :retrieval_count
    ]

    candidate_indices = candidate_indices[
        np.argsort(
            -similarity_scores[
                candidate_indices
            ],
            kind="stable",
        )
    ]

    recommendations = []
    seen_appids = {
        query_appid
    }

    optional_columns = [
        "release_date",
        "release_year",
        "genres",
        "recommendations",
        "pct_pos_total",
        "num_reviews_total",
        "peak_ccu",
    ]

    for candidate_model_row in candidate_indices:
        candidate_row = (
            semantic_games.iloc[
                int(
                    candidate_model_row
                )
            ]
        )

        candidate_appid = (
            candidate_row[
                "appid"
            ]
        )

        if candidate_appid in seen_appids:
            continue

        seen_appids.add(
            candidate_appid
        )

        semantic_score = float(
            similarity_scores[
                candidate_model_row
            ]
        )

        recommendation = {
            "appid": candidate_appid,
            "recommended_game": (
                candidate_row[
                    "display_name"
                ]
            ),
            "semantic_score": round(
                max(
                    -1.0,
                    min(
                        1.0,
                        semantic_score,
                    ),
                ),
                6,
            ),
        }

        for column in optional_columns:
            if column in semantic_games.columns:
                recommendation[
                    column
                ] = candidate_row[
                    column
                ]

        recommendations.append(
            recommendation
        )

        if (
            len(recommendations)
            >= top_n
        ):
            break

    result = pd.DataFrame(
        recommendations
    )

    if result.empty:
        return result

    sort_columns = [
        "semantic_score",
    ]

    ascending = [
        False,
    ]

    if "recommendations" in result.columns:
        sort_columns.append(
            "recommendations"
        )

        ascending.append(
            False
        )

    result = (
        result
        .sort_values(
            sort_columns,
            ascending=ascending,
            na_position="last",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    result.insert(
        0,
        "semantic_rank",
        np.arange(
            1,
            len(result) + 1,
        ),
    )

    return result

## 9. Recommendation Case Studies

The same four queries used by the metadata and rich-text TF-IDF notebooks are retained:

- **ELDEN RING**
- **Red Dead Redemption 2**
- **PUBG: BATTLEGROUNDS**
- **Assassin's Creed Odyssey**

Using identical queries allows qualitative comparison between lexical, semantic, structured and later hybrid rankings.

In [20]:
development_queries = [
    "ELDEN RING",
    "Red Dead Redemption 2",
    "PUBG: BATTLEGROUNDS",
    "Assassin's Creed Odyssey",
]

for game_title in development_queries:
    print(
        "\n" + "=" * 80
    )

    print(
        f"Semantic recommendations for: {game_title}"
    )

    print(
        "=" * 80
    )

    display(
        recommend_semantic_v1(
            game_title=game_title,
            top_n=10,
            candidate_pool=50,
        )
    )


Semantic recommendations for: ELDEN RING


,semantic_rank,appid,recommended_game,semantic_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1,3143110,Elden Sword,0.713858,2024-08-19,2024,"['Action', 'Adventure', 'Casual', 'Indie']",0,NaN,0.0,0
1,2,2392310,エルミナージュORIGINAL ～闇の巫女と神々の指輪～,0.660008,2024-05-23,2024,['RPG'],0,79.0,86.0,5
2,3,593410,Elven Legend,0.657197,2017-03-08,2017,"['Casual', 'Indie', 'Strategy']",0,73.0,42.0,0
3,4,737490,Journey to Luonto,0.656227,2017-11-27,2017,"['Adventure', 'Free To Play', 'RPG']",0,78.0,28.0,0
4,5,2120790,Ringborn,0.654274,2023-01-12,2023,"['Action', 'Adventure', 'RPG']",0,NaN,0.0,0
5,6,2149990,Gems Defenders,0.651456,2024-08-30,2024,"['Action', 'Adventure', 'Casual', 'Indie', 'RP...",0,NaN,0.0,0
6,7,2493560,Heros and Monsters: Idle Incremental,0.651438,2023-07-21,2023,"['Action', 'Adventure', 'Casual', 'Indie', 'Si...",0,69.0,105.0,0
7,8,1501750,Lords of the Fallen,0.645743,2023-10-13,2023,"['Action', 'Adventure', 'RPG']",23921,63.0,23930.0,365
8,9,2699890,Battlecry Berserkers,0.644013,2024-09-06,2024,"['Action', 'Adventure', 'Indie', 'RPG']",0,NaN,0.0,0
9,10,581340,Narborion Saga,0.642581,2017-03-20,2017,"['Adventure', 'RPG', 'Strategy']",0,68.0,29.0,28



Semantic recommendations for: Red Dead Redemption 2


,semantic_rank,appid,recommended_game,semantic_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1,1404210,Red Dead Online,0.760392,2020-12-01,2020,"['Action', 'Adventure']",56631,83.0,56651.0,1228
1,2,2668510,Red Dead Redemption,0.718439,2024-10-29,2024,['Action'],8276,92.0,8295.0,763
2,3,39800,Nation Red,0.617577,2009-09-03,2009,"['Action', 'Indie']",2166,88.0,2166.0,2
3,4,312050,Red Johnson's Chronicles - 1+2 - Steam Special...,0.603020,2014-09-05,2014,"['Action', 'Adventure']",0,39.0,23.0,0
4,5,916320,Fair Deal: Las Vegas,0.598599,2018-12-21,2018,"['Action', 'Indie']",0,100.0,12.0,0
5,6,2753270,Deadrock Redemption,0.595672,2024-04-04,2024,"['Adventure', 'RPG']",215,65.0,216.0,1
6,7,2933080,Crime Boss: Rockay City,0.594591,2024-06-18,2024,['Action'],4831,72.0,4840.0,153
7,8,709870,West of Red,0.589464,2018-03-07,2018,"['Action', 'Indie', 'Simulation']",0,52.0,97.0,0
8,9,2439740,Wolf West,0.588975,2023-09-21,2023,"['Adventure', 'Casual', 'Free To Play', 'Indie']",0,84.0,13.0,0
9,10,598550,HUNTDOWN,0.588876,2021-05-12,2021,"['Action', 'Indie']",3313,94.0,3319.0,17



Semantic recommendations for: PUBG: BATTLEGROUNDS


,semantic_rank,appid,recommended_game,semantic_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1,725620,Battlegrounds2D.IO,0.648040,2019-08-15,2019,['Action'],0,NaN,0.0,0
1,2,2630220,HEROS FIGHT Battle royal,0.639223,2023-10-27,2023,"['Action', 'Strategy']",0,NaN,0.0,0
2,3,873220,SurvivalZ Battlegrounds,0.618250,2019-04-02,2019,"['Action', 'Indie', 'Simulation', 'Strategy']",273,64.0,273.0,0
3,4,1313090,WWII: Rising,0.598685,2020-10-15,2020,"['Action', 'Free To Play', 'Indie', 'Massively...",0,44.0,83.0,0
4,5,1797680,Wee Tanks!,0.592686,2023-05-16,2023,"['Action', 'Casual', 'Indie', 'Strategy']",436,91.0,436.0,5
5,6,1148300,War Battle Royale Battlegrounds,0.588375,2019-09-11,2019,"['Action', 'Simulation', 'Strategy']",0,NaN,0.0,0
6,7,3131780,Legends' Battlegrounds,0.586792,2025-01-23,2025,"['Action', 'Indie', 'Early Access']",0,NaN,0.0,0
7,8,823940,Virtual Battlegrounds,0.585812,2020-04-15,2020,"['Action', 'Adventure', 'Indie', 'Massively Mu...",404,66.0,404.0,0
8,9,2305790,1v1.LOL - Battle Royale Game,0.585739,2023-06-18,2023,"['Action', 'Adventure', 'Casual', 'Massively M...",0,74.0,12017.0,0
9,10,3166520,Frontier: Battle Royale,0.585212,2024-09-20,2024,"['Action', 'Indie', 'Free To Play']",0,59.0,22.0,0



Semantic recommendations for: Assassin's Creed Odyssey


,semantic_rank,appid,recommended_game,semantic_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1,2251680,Epic Assassin,0.744521,2022-12-30,2022,['RPG'],0,NaN,0.0,0
1,2,15100,Assassin's Creed™: Director's Cut Edition,0.705924,2008-04-09,2008,"['Action', 'Adventure']",19551,81.0,19777.0,84
2,3,1355070,Aenaon,0.705891,2020-07-26,2020,"['Adventure', 'Early Access']",0,NaN,0.0,0
3,4,208750,Apotheon,0.698247,2015-02-03,2015,"['Action', 'Adventure', 'Indie', 'RPG']",3158,90.0,3164.0,5
4,5,201870,Assassin's Creed® Revelations,0.695097,2011-11-29,2011,"['Action', 'Adventure']",15923,84.0,16035.0,174
5,6,582160,Assassin's Creed® Origins,0.693502,2017-10-26,2017,"['Action', 'Adventure', 'RPG']",102229,85.0,102363.0,3931
6,7,2059530,Dawn of Defiance,0.685921,2024-08-15,2024,"['Action', 'Adventure', 'Early Access']",1291,77.0,1291.0,29
7,8,1490180,For Sparta,0.680421,2021-03-22,2021,['Action'],0,81.0,22.0,0
8,9,1533070,Oracle Trials,0.672837,2021-02-12,2021,"['Action', 'Casual']",0,90.0,10.0,0
9,10,540690,Hellenica,0.666660,2017-01-23,2017,"['Indie', 'RPG', 'Strategy']",0,65.0,44.0,0


## 10. Preparing Semantic Candidates for the Hybrid Recommender

For each shared query, the Top 50 semantic candidates are generated using the same output structure as the other candidate-generation models.

The CSV is used for hybrid development and evaluation. In the final live system, the reusable semantic recommendation function can be called directly for any semantic-eligible catalogue game.

In [22]:
def generate_semantic_candidates(
    query_title,
    top_n=50,
    candidate_pool=100,
):
    """Generate a hybrid-compatible semantic candidate table."""

    results = recommend_semantic_v1(
        game_title=query_title,
        top_n=top_n,
        candidate_pool=candidate_pool,
    ).copy()

    title_key = normalise_game_title(
        query_title
    )

    query_model_row = (
        semantic_title_to_model_row[
            title_key
        ]
    )

    query_row = semantic_games.iloc[
        query_model_row
    ]

    results.insert(
        0,
        "query_appid",
        query_row[
            "appid"
        ],
    )

    results.insert(
        1,
        "query_game",
        query_row[
            "display_name"
        ],
    )

    return results

In [24]:
semantic_candidate_tables = []

for game_title in development_queries:
    print(
        f"Generating semantic candidates for: {game_title}"
    )

    candidates = generate_semantic_candidates(
        query_title=game_title,
        top_n=50,
        candidate_pool=100,
    )

    semantic_candidate_tables.append(
        candidates
    )

semantic_candidates = pd.concat(
    semantic_candidate_tables,
    ignore_index=True,
)

print(
    "Combined semantic candidate shape:",
    semantic_candidates.shape,
)

display(
    semantic_candidates[
        [
            "query_game",
            "semantic_rank",
            "appid",
            "recommended_game",
            "semantic_score",
        ]
    ]
    .head(20)
)

Generating semantic candidates for: ELDEN RING
Generating semantic candidates for: Red Dead Redemption 2
Generating semantic candidates for: PUBG: BATTLEGROUNDS
Generating semantic candidates for: Assassin's Creed Odyssey
Combined semantic candidate shape: (200, 13)


,query_game,semantic_rank,appid,recommended_game,semantic_score
0,ELDEN RING,1,3143110,Elden Sword,0.713858
1,ELDEN RING,2,2392310,エルミナージュORIGINAL ～闇の巫女と神々の指輪～,0.660008
2,ELDEN RING,3,593410,Elven Legend,0.657197
3,ELDEN RING,4,737490,Journey to Luonto,0.656227
4,ELDEN RING,5,2120790,Ringborn,0.654274
5,ELDEN RING,6,2149990,Gems Defenders,0.651456
6,ELDEN RING,7,2493560,Heros and Monsters: Idle Incremental,0.651438
7,ELDEN RING,8,1501750,Lords of the Fallen,0.645743
8,ELDEN RING,9,2699890,Battlecry Berserkers,0.644013
9,ELDEN RING,10,581340,Narborion Saga,0.642581


## 11. Candidate Validation and Export

Every query must contain 50 candidates. The exported handoff must contain no missing identifiers, duplicate query-candidate pairs, self-recommendations, invalid ranks or non-finite semantic scores.

In [25]:
query_counts = (
    semantic_candidates
    .groupby(
        "query_game"
    )
    .size()
)

duplicate_pairs = (
    semantic_candidates
    .duplicated(
        subset=[
            "query_appid",
            "appid",
        ]
    )
    .sum()
)

missing_appids = (
    semantic_candidates[
        "appid"
    ]
    .isna()
    .sum()
)

self_recommendations = (
    semantic_candidates[
        "query_appid"
    ]
    .eq(
        semantic_candidates[
            "appid"
        ]
    )
    .sum()
)

invalid_scores = (
    ~semantic_candidates[
        "semantic_score"
    ]
    .between(
        -1,
        1,
        inclusive="both",
    )
).sum()

non_finite_scores = (
    ~np.isfinite(
        semantic_candidates[
            "semantic_score"
        ]
        .to_numpy(
            dtype=float
        )
    )
).sum()

invalid_rank_groups = 0

for _, group in (
    semantic_candidates
    .groupby(
        "query_game"
    )
):
    actual_ranks = (
        group
        .sort_values(
            "semantic_rank"
        )[
            "semantic_rank"
        ]
        .tolist()
    )

    expected_ranks = list(
        range(
            1,
            len(group) + 1,
        )
    )

    if (
        actual_ranks
        != expected_ranks
    ):
        invalid_rank_groups += 1

expected_rows = (
    len(development_queries) * 50
)

print(
    "Candidates per query:"
)

print(
    query_counts
)

print(
    "\nCombined candidate shape:",
    semantic_candidates.shape,
)

print(
    "Duplicate query-candidate pairs:",
    duplicate_pairs,
)

print(
    "Missing candidate appids:",
    missing_appids,
)

print(
    "Self-recommendations:",
    self_recommendations,
)

print(
    "Invalid scores:",
    invalid_scores,
)

print(
    "Non-finite scores:",
    non_finite_scores,
)

print(
    "Query groups with invalid ranks:",
    invalid_rank_groups,
)

assert len(
    semantic_candidates
) == expected_rows

assert (
    query_counts == 50
).all()

assert duplicate_pairs == 0
assert missing_appids == 0
assert self_recommendations == 0
assert invalid_scores == 0
assert non_finite_scores == 0
assert invalid_rank_groups == 0

print(
    "\nSemantic candidate validation passed."
)

Candidates per query:
query_game
Assassin's Creed® Odyssey    50
ELDEN RING                   50
PUBG: BATTLEGROUNDS          50
Red Dead Redemption 2        50
dtype: int64

Combined candidate shape: (200, 13)
Duplicate query-candidate pairs: 0
Missing candidate appids: 0
Self-recommendations: 0
Invalid scores: 0
Non-finite scores: 0
Query groups with invalid ranks: 0

Semantic candidate validation passed.


In [38]:
candidate_output_path = os.path.join(
    OUTPUT_DIR,
    "semantic_candidates.csv",
)

config_output_path = os.path.join(
    OUTPUT_DIR,
    "semantic_model_config.json",
)

semantic_candidates.to_csv(
    candidate_output_path,
    index=False,
)

semantic_model_config = {
    "model_name": MODEL_NAME,
    "sentence_transformers_version": (
        REQUIRED_SENTENCE_TRANSFORMERS_VERSION
    ),
    "embedding_dimension": int(
        semantic_embeddings.shape[1]
    ),
    "embedding_dtype": str(
        semantic_embeddings.dtype
    ),
    "embedding_normalisation": "L2",
    "similarity_metric": "cosine_via_dot_product",
    "primary_text_field": (
        "about_the_game_clean"
    ),
    "fallback_text_field": (
        "short_description_clean"
    ),
    "minimum_description_words": (
        MIN_DESCRIPTION_WORDS
    ),
    "maximum_semantic_words": (
        MAX_SEMANTIC_WORDS
    ),
    "excluded_input_fields": [
        "display_name",
        "genres",
        "tags",
        "categories",
        "developers",
        "publishers",
        "release_date",
        "recommendations",
        "review_variables",
        "peak_ccu",
    ],
    "batch_size": BATCH_SIZE,
    "encoding_device": DEVICE,
    "candidate_pool": 100,
    "exported_candidates_per_query": 50,
    "case_study_queries": development_queries,
}

with open(
    config_output_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        semantic_model_config,
        file,
        indent=4,
    )

print(
    "Semantic candidates saved to:\n"
    f"{candidate_output_path}"
)

print(
    "\nSemantic embeddings saved to:\n"
    f"{embedding_output_path}"
)

print(
    "\nSemantic index saved to:\n"
    f"{embedding_index_output_path}"
)

print(
    "\nModel configuration saved to:\n"
    f"{config_output_path}"
)

Semantic candidates saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/semantic_outputs/semantic_candidates.csv

Semantic embeddings saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/semantic_outputs/semantic_embeddings_all_mpnet_base_v2.npy

Semantic index saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/semantic_outputs/semantic_embedding_index_all_mpnet_base_v2.csv

Model configuration saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/semantic_outputs/semantic_model_config.json


In [39]:
saved_semantic_candidates = pd.read_csv(
    candidate_output_path
)

saved_semantic_index = pd.read_csv(
    embedding_index_output_path
)

saved_semantic_embeddings = np.load(
    embedding_output_path,
    mmap_mode="r",
)

print(
    "Saved candidate shape:",
    saved_semantic_candidates.shape,
)

print(
    "Saved embedding shape:",
    saved_semantic_embeddings.shape,
)

print(
    "Saved index shape:",
    saved_semantic_index.shape,
)

display(
    saved_semantic_candidates.head(10)
)

Saved candidate shape: (200, 13)
Saved embedding shape: (87568, 768)
Saved index shape: (87568, 4)


,query_appid,query_game,semantic_rank,appid,recommended_game,semantic_score,release_date,release_year,genres,recommendations,pct_pos_total,num_reviews_total,peak_ccu
0,1245620,ELDEN RING,1,3143110,Elden Sword,0.713858,2024-08-19,2024,"['Action', 'Adventure', 'Casual', 'Indie']",0,NaN,0.0,0
1,1245620,ELDEN RING,2,2392310,エルミナージュORIGINAL ～闇の巫女と神々の指輪～,0.660008,2024-05-23,2024,['RPG'],0,79.0,86.0,5
2,1245620,ELDEN RING,3,593410,Elven Legend,0.657197,2017-03-08,2017,"['Casual', 'Indie', 'Strategy']",0,73.0,42.0,0
3,1245620,ELDEN RING,4,737490,Journey to Luonto,0.656227,2017-11-27,2017,"['Adventure', 'Free To Play', 'RPG']",0,78.0,28.0,0
4,1245620,ELDEN RING,5,2120790,Ringborn,0.654274,2023-01-12,2023,"['Action', 'Adventure', 'RPG']",0,NaN,0.0,0
5,1245620,ELDEN RING,6,2149990,Gems Defenders,0.651456,2024-08-30,2024,"['Action', 'Adventure', 'Casual', 'Indie', 'RP...",0,NaN,0.0,0
6,1245620,ELDEN RING,7,2493560,Heros and Monsters: Idle Incremental,0.651438,2023-07-21,2023,"['Action', 'Adventure', 'Casual', 'Indie', 'Si...",0,69.0,105.0,0
7,1245620,ELDEN RING,8,1501750,Lords of the Fallen,0.645743,2023-10-13,2023,"['Action', 'Adventure', 'RPG']",23921,63.0,23930.0,365
8,1245620,ELDEN RING,9,2699890,Battlecry Berserkers,0.644013,2024-09-06,2024,"['Action', 'Adventure', 'Indie', 'RPG']",0,NaN,0.0,0
9,1245620,ELDEN RING,10,581340,Narborion Saga,0.642581,2017-03-20,2017,"['Adventure', 'RPG', 'Strategy']",0,68.0,29.0,28


## 15. Findings and Limitations

The transformer model is expected to identify games that express related settings, mechanics, themes or experiences even when they use different vocabulary. This provides a different signal from lexical TF-IDF and structured semantic overlap.

However, the model remains a general-purpose encoder rather than a game-domain model. It may interpret promotional prose imperfectly, and semantically similar language does not guarantee identical gameplay. Long descriptions are limited to their first 250 words, so later details may be omitted. Description quality and catalogue completeness also remain important.

The model intentionally avoids appending titles and structured semantic. This reduces duplication with the metadata recommender, but it also means that games with weak descriptions cannot be rescued by tag or genre information inside this component. That separation is deliberate because the hybrid model will combine complementary signals later.

## 16. Conclusion

This notebook implemented a transformer-based semantic game recommender using `sentence-transformers/all-mpnet-base-v2`.

A controlled semantic text field was created from cleaned Steam descriptions only. `about_the_game_clean` was used as the primary field, with `short_description_clean` as a fallback. Titles, genres, tags, categories, developer names, publisher names and engagement variables were not appended to the model input.

Descriptions satisfying the minimum-word rule were converted into normalised 768-dimensional dense embeddings. CUDA was used automatically when available, and the saved embedding matrix enabled cosine retrieval through efficient dot products.

The final function returns standardised semantic ranks and scores. Top 50 candidates for the four shared case-study games were exported as `semantic_candidates.csv`. These candidates will form the semantic component of the later weighted hybrid recommender.

## Evaluation Candidate Generation

For formal model evaluation, the final semantic recommender is applied to ten representative query games covering different genres and gameplay styles.

The model generates 50 candidates per query so that the same candidate pool can later be used by the hybrid recommender. Only the Top 10 results will be used when calculating evaluation metrics.

In [40]:
evaluation_games = [
    "Counter-Strike 2",
    "Grand Theft Auto V Legacy",
    "Stardew Valley",
    "Sid Meier's Civilization VI",
    "Hollow Knight",
    "The Witcher 3: Wild Hunt",
    "Hades",
    "Deep Rock Galactic",
    "Factorio",
    "Phasmophobia"
]

print("Number of evaluation games:", len(evaluation_games))

Number of evaluation games: 10


In [42]:
import time

semantic_evaluation_tables = []
semantic_runtime_records = []

for game_title in evaluation_games:

    print("Generating semantic candidates for:", game_title)

    start_time = time.perf_counter()

    candidates = generate_semantic_candidates(
        query_title=game_title,
        top_n=50,
        candidate_pool=100
    )

    end_time = time.perf_counter()

    query_runtime = end_time - start_time

    semantic_evaluation_tables.append(
        candidates
    )

    runtime_record = {
        "model": "semantic",
        "query_game": game_title,
        "runtime_seconds": query_runtime
    }

    semantic_runtime_records.append(
        runtime_record
    )

semantic_evaluation_candidates = pd.concat(
    semantic_evaluation_tables,
    ignore_index=True
)

semantic_runtime_results = pd.DataFrame(
    semantic_runtime_records
)

print(
    "\nCandidate dataset shape:",
    semantic_evaluation_candidates.shape
)

print(
    "Runtime dataset shape:",
    semantic_runtime_results.shape
)

Generating semantic candidates for: Counter-Strike 2
Generating semantic candidates for: Grand Theft Auto V Legacy
Generating semantic candidates for: Stardew Valley
Generating semantic candidates for: Sid Meier's Civilization VI
Generating semantic candidates for: Hollow Knight
Generating semantic candidates for: The Witcher 3: Wild Hunt
Generating semantic candidates for: Hades
Generating semantic candidates for: Deep Rock Galactic
Generating semantic candidates for: Factorio
Generating semantic candidates for: Phasmophobia

Candidate dataset shape: (500, 13)
Runtime dataset shape: (10, 3)


## Evaluation Candidate Validation and Export
The generated semantic candidates are checked for completeness, duplicate recommendations and accidental inclusion of the query game.

The validated candidate results and query runtimes are then exported for use in the separate evaluation notebook.

In [31]:
# Check the number of query games.
query_count = semantic_evaluation_candidates[
    "query_game"
].nunique()

# Check the number of candidates for each query game.
candidate_counts = semantic_evaluation_candidates.groupby(
    "query_game"
).size()

# Check duplicate recommendations within each query.
duplicate_count = semantic_evaluation_candidates.duplicated(
    subset=[
        "query_appid",
        "appid"
    ]
).sum()

# Check whether a query game recommends itself.
self_recommendations = semantic_evaluation_candidates[
    semantic_evaluation_candidates["query_appid"]
    ==
    semantic_evaluation_candidates["appid"]
]

print("Number of query games:", query_count)
print("\nCandidates per query:")
print(candidate_counts)

print(
    "\nDuplicate recommendations:",
    duplicate_count
)

print(
    "Self-recommendations:",
    len(self_recommendations)
)

Number of query games: 10

Candidates per query:
query_game
Counter-Strike 2                50
Deep Rock Galactic              50
Factorio                        50
Grand Theft Auto V Legacy       50
Hades                           50
Hollow Knight                   50
Phasmophobia                    50
Sid Meier’s Civilization® VI    50
Stardew Valley                  50
The Witcher 3: Wild Hunt        50
dtype: int64

Duplicate recommendations: 0
Self-recommendations: 0


In [32]:
# Define output file paths.

semantic_evaluation_path = (
    "/content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/"
    "semantic_evaluation_candidates.csv"
)

semantic_runtime_path = (
    "/content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/"
    "semantic_evaluation_runtime.csv"
)

# Save evaluation candidates.
semantic_evaluation_candidates.to_csv(
    semantic_evaluation_path,
    index=False
)

# Save runtime results.
semantic_runtime_results.to_csv(
    semantic_runtime_path,
    index=False
)

print(
    "Candidate file saved to:",
    semantic_evaluation_path
)

print(
    "Runtime file saved to:",
    semantic_runtime_path
)

Candidate file saved to: /content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/semantic_evaluation_candidates.csv
Runtime file saved to: /content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/semantic_evaluation_runtime.csv
